# Day 30 — A/B Test Design & Analysis
### Statistics & Probability · Module 2 · Topic 2.19

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · 90 min concepts + 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | What an A/B test is, and what randomising buys you | 15 min |
| 2 | **Size it before you run it** — MDE, power, the winner's curse | 20 min |
| 3 | ⚠️ **Running it without breaking it** — peeking, SRM, novelty | 20 min |
| 4 | Analysing it — the test, revenue, segments | 20 min |
| 5 | Deciding and reporting | 15 min |
| 6 | **Practice notebook (separate file)** | 30 min |

---

## The step we take today

Yesterday ended with five explanations for a correlation: chance, confounding, reverse
causation, selection — and cause. With observational data you can argue about which one it
is, and you can never settle it.

Today you get the one design that settles it. **A coin flip decides who sees which
version.** That single act switches off confounding, reverse causation and selection at the
same time, and leaves only chance — which is exactly what the whole of Module 2 has been
teaching you to handle.

### So why does this need two hours?

Because the analysis is five lines of code, and almost everything that goes wrong with A/B
tests happens *before* the analysis or *around* it. The maths is easy. Running the test
honestly is not.

### Four claims to hold me to

1. The same users give a **+21** effect observationally and **+2.4** when randomised.
2. A small test that "wins" overstates the true lift by **3.3×** — with a significant p-value.
3. Checking the dashboard 20 times turns a 5% error rate into **24%**.
4. A real **+10%** revenue lift has **9%** power at n = 1,000 per arm. You would miss it.

This is also the **last session of Module 2**. Everything from Day 23 onwards gets used here
at once.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

plt.rcParams["figure.figsize"] = (8, 3.2)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", __import__("scipy").__version__)

---
# 1 · What an A/B test is

## 1.1 · ⭐ The coin flip is the entire trick

Here is the same population of users, measured two ways.

- **Observationally:** some users adopted a new feature on their own. Compare adopters with
  non-adopters.
- **Randomised:** a coin decided who got the feature. Compare the two arms.

In this simulation the feature's true effect is **+2.0** in spend, and engaged users are both
more likely to adopt *and* higher spenders regardless.

In [ ]:
rng = np.random.default_rng(3100)
N = 600

engagement = rng.normal(0, 1, N)                 # the confounder, unobserved in real life
adopts     = (engagement + rng.normal(0, .5, N)) > 0.4
spend_obs  = 40 + 14*engagement + rng.normal(0, 6, N)      # note: adoption has NO effect here

arm       = rng.random(N) < 0.5                  # a coin, not a choice
spend_rnd = 40 + 14*engagement + 2.0*arm + rng.normal(0, 6, N)   # true effect = +2.0

print("OBSERVED   adopters %.1f  vs  non-adopters %.1f   -> gap %+.1f"
      % (spend_obs[adopts].mean(), spend_obs[~adopts].mean(),
         spend_obs[adopts].mean() - spend_obs[~adopts].mean()))
print("RANDOMISED treatment %.1f  vs  control %.1f       -> gap %+.1f"
      % (spend_rnd[arm].mean(), spend_rnd[~arm].mean(),
         spend_rnd[arm].mean() - spend_rnd[~arm].mean()))
print("\nTRUE effect of the feature, by construction: +2.0")

The observational comparison says **+21**. The feature does nothing at all in that simulation
— the gap is entirely engagement, which drives both adoption and spend. It is wrong by a
factor of ten.

The randomised comparison says **+2.4** against a truth of **+2.0**. Not exact — the coin
balances things *on average*, not in this particular draw — but it is a third of a standard
error away, and crucially it is **unbiased**: run the experiment again and again and the
average estimate converges on +2.0.

In [ ]:
# the standard error of that randomised difference
gap = spend_rnd[arm].mean() - spend_rnd[~arm].mean()
se  = np.sqrt(spend_rnd[arm].var(ddof=1)/arm.sum() + spend_rnd[~arm].var(ddof=1)/(~arm).sum())
print("randomised estimate %+.2f   true effect +2.00" % gap)
print("SE of the difference %.2f" % se)
print("-> the estimate is %.1f standard errors from the truth" % (abs(2.0-gap)/se))

print()
print("and the observational estimate, for comparison:")
print("   %+.1f  -> %.0f standard errors from the truth"
      % (spend_obs[adopts].mean()-spend_obs[~adopts].mean(),
         abs(2.0-(spend_obs[adopts].mean()-spend_obs[~adopts].mean()))/se))

## 1.2 · What the coin fixes, and what it does not

| | |
|---|---|
| ✅ **Confounding** | every variable — including the ones you never measured or thought of — is balanced on average |
| ✅ **Reverse causation** | the coin came first, in time, so the outcome cannot have caused the assignment |
| ✅ **Selection** | you decide who is in the test before anyone has an outcome |
| ❌ **Chance** | the coin balances on average, not in *your* test |
| ❌ **Too small a test** | randomising badly-sized arms answers nothing, correctly |
| ❌ **The wrong metric** | a clean test of something nobody cares about |
| ❌ **Too short a horizon** | you measured a week and shipped for a year |

> **"Balanced on average" is the exact claim.** In any one test the groups will differ by
> chance. That difference is precisely the uncertainty the p-value and the confidence
> interval are measuring — it is not a flaw in the design, it is the thing the maths is for.

## 1.3 · Three decisions, all made before the test starts

**1. The unit — what gets randomised.** Almost always the **user**, not the session and
never the pageview. Randomise by pageview and the same person sees both versions (they
notice), and their pageviews are not independent, so every p-value you compute is wrong.

**2. The metric — one primary metric, written down first.** Everything else is a secondary
metric or a guardrail. Secondary metrics cannot *win* the test; they can only lose it.

**3. The size — worked out from the smallest effect worth shipping.** That is section 2.

Write all three in a one-page plan and have someone sign it. After the data arrives every
one of these becomes negotiable, and that is how tests get p-hacked without anyone deciding
to cheat.

---
# 2 · Size it before you run it

## 2.1 · The question is not "how many users?"

It is **"how small an effect do I need to be able to catch?"** — the *minimum detectable
effect*, or MDE. That is a business question, not a statistical one: what is the smallest
lift that would make you ship this?

Everything else follows mechanically.

In [ ]:
def n_per_arm(p0, mde, power=0.80, alpha=0.05):
    """Users needed in EACH arm of a two-sided two-proportion test."""
    p1   = p0 + mde
    pbar = (p0 + p1) / 2
    za   = stats.norm.ppf(1 - alpha/2)       # 1.96
    zb   = stats.norm.ppf(power)             # 0.84
    num  = (za*np.sqrt(2*pbar*(1-pbar)) +
            zb*np.sqrt(p0*(1-p0) + p1*(1-p1)))**2
    return int(np.ceil(num / mde**2))

print(f"{'MDE':>8} | {'relative':>9} | {'n per arm':>10} | {'days @ 2,000/day/arm':>21}")
print("-"*58)
for mde in [0.005, 0.0075, 0.01, 0.02, 0.03, 0.05]:
    n = n_per_arm(0.10, mde)
    print(f"  +{mde*100:>4.2f}pp | {mde/0.10:>8.0%} | {n:>10,} | {n/2000:>20.1f}")

**The quadrupling rule.** n scales as 1/MDE², so halving the effect you want to catch costs
roughly **four times** the users. Catching +2pp takes 3,841 per arm and two days. Catching
+0.5pp takes 57,763 per arm and a month.

This is why "let's just run it and see" is so expensive. If you have not decided what size
of effect matters, you cannot know whether a month of traffic is enough or ten times too
much.

In [ ]:
# Cross-check against statsmodels, if it is installed
try:
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.proportion import proportion_effectsize
    print(f"{'MDE':>8} | {'ours':>8} | {'statsmodels':>12}")
    print("-"*34)
    for mde in [0.01, 0.02, 0.05]:
        es = proportion_effectsize(0.10 + mde, 0.10)
        n_sm = NormalIndPower().solve_power(es, power=0.8, alpha=0.05, ratio=1)
        print(f"  +{mde*100:>4.1f}pp | {n_per_arm(0.10, mde):>8,} | {int(np.ceil(n_sm)):>12,}")
    print("\nDifferences of ~0.04% come from a different variance convention. Either is fine.")
except ImportError:
    print("statsmodels not installed - pip install statsmodels")

Write it yourself once, so you know what each input means. Then use the library.

## 2.2 · ⚠️ The winner's curse

An underpowered test does not simply fail to find things. **When it does find something, it
overstates it** — and it is still "significant".

Below, the true lift is **+1.00pp** on a 10% baseline in all three cases. We run thousands of
tests at each sample size and look only at the ones that "won" (p < 0.05 in the right
direction), which is exactly what you would see in real life.

In [ ]:
def two_prop(c_a, n_a, c_b, n_b, alpha=0.05):
    p_a, p_b = c_a/n_a, c_b/n_b
    diff = p_b - p_a
    pbar = (c_a + c_b)/(n_a + n_b)
    z = diff / np.sqrt(pbar*(1-pbar)*(1/n_a + 1/n_b))
    p = 2*(1 - stats.norm.cdf(abs(z)))
    se = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
    c = stats.norm.ppf(1 - alpha/2)
    return diff, p, (diff - c*se, diff + c*se)

rng = np.random.default_rng(3001)
TRUE_A, TRUE_B, TRIALS = 0.10, 0.11, 4000

print(f"{'n per arm':>10} | {'power':>6} | {'avg lift among winners':>23} | inflation")
print("-"*62)
for n in [1000, 3000, 14751]:
    lifts, n_sig = [], 0
    for _ in range(TRIALS):
        a = rng.binomial(n, TRUE_A); b = rng.binomial(n, TRUE_B)
        diff, p, _ = two_prop(a, n, b, n)
        if p < 0.05 and diff > 0:
            n_sig += 1; lifts.append(diff*100)
    print(f"{n:>10,} | {n_sig/TRIALS:>5.0%}  | {np.mean(lifts):>+21.2f}pp | {np.mean(lifts)/1.0:>6.2f}x")
print("\n(the truth is +1.00pp in every row)")

At **n = 1,000** the test has about 11% power. The only tests that reach significance are the
ones where noise happened to push the gap up, so the winners average **+3.32pp** — a
**3.3× overstatement**, with p < 0.05 attached to it.

This is a large part of why shipped features so often fail to deliver the lift their test
promised. It is not fraud and it is not a broken test; it is arithmetic.

Notice the last row: at n = 14,751 — the number our formula produced for 80% power — the
simulation measures **80% power** and the inflation falls to **1.12×**. That is an
independent check on the sample-size formula, and it is the real argument for powering tests
properly: **power is not just about finding effects, it is about measuring them.**

---
# 3 · ⚠️ Running it without breaking it

## 3.1 · Peeking

The p-value wanders as data arrives. If you stop the moment it dips below 0.05, you have
given yourself many chances to catch a fluctuation instead of one.

Below, there is **no real effect at all**. Both arms convert at exactly 10%.

In [ ]:
rng = np.random.default_rng(3002)
TRIALS = 3000
N_FINAL = 14751

print(f"{'looks':>6} | {'false win rate':>15}")
print("-"*25)
for peeks in [1, 2, 5, 10, 20]:
    step, hits = N_FINAL//peeks, 0
    for _ in range(TRIALS):
        ca = cb = tot = 0
        for _k in range(peeks):
            ca += rng.binomial(step, 0.10); cb += rng.binomial(step, 0.10)
            tot += step
            if two_prop(ca, tot, cb, tot)[1] < 0.05:
                hits += 1; break
    print(f"{peeks:>6} | {hits/TRIALS:>14.1%}")

One look: 4%, as designed. Twenty looks: **25%** — one test in four declares a winner on
data with nothing in it.

Nobody sets out to do this. It is simply what happens when a dashboard exists and somebody
is excited.

**The three honest options:**

1. **Fix the end date** from the sample-size sum, and do not look at the primary metric.
2. **Look, but only at guardrails** — crashes, error rates, refunds. You may stop a test for
   *harm*; you may not stop it for *success*.
3. **Use a method designed for it** — sequential testing, group-sequential boundaries, or
   always-valid p-values. These exist and they work. A plain z-test checked daily is not one
   of them.

## 3.2 · Sample ratio mismatch (SRM)

You asked for a 50/50 split. Did you get one? A chi-square test on the two counts answers it
in one line, and it is the highest-value five minutes in the whole process.

In [ ]:
print(f"{'split':>22} | {'chi2':>8} | {'p':>10} | verdict")
print("-"*60)
for na, nb in [(20000, 20000), (20100, 19900), (20250, 19750), (20600, 19400)]:
    tot = na + nb
    res = stats.chisquare([na, nb], [tot/2, tot/2])
    flag = "fine" if res.pvalue > 0.01 else "*** BUG ***"
    print(f"{na:>9,} / {nb:<9,} | {res.statistic:>8.2f} | {res.pvalue:>10.2e} | {flag}")

A **51.5 / 48.5** split on 40,000 users gives p = 2e-09. That is not bad luck; something
dropped users from one arm.

⚠️ **An SRM invalidates the entire test**, not just the split. Users were lost unevenly — a
redirect that timed out, a bot filter that hit one variant, logging that dropped slow
sessions. Whatever was lost was *not random*, so the two groups are no longer comparable and
every number downstream is suspect.

The correct response is to fix the bug and rerun. Do not analyse it, and do not "adjust" for
it.

## 3.3 · ⚠️ Novelty and the weekend

Two time effects, both of which will fool you.

In [ ]:
days    = np.arange(1, 29)
base    = np.array([0.118, 0.116, 0.112, 0.110, 0.104, 0.092, 0.090]*4)  # weekday/weekend
novelty = 0.025*np.exp(-days/3.0)                                        # fades over ~a week
treat   = base + novelty

print("weekday baseline %.1f%%   weekend baseline %.1f%%   swing %.1fpp"
      % (base[:5].mean()*100, base[5:7].mean()*100, (base[:5].mean()-base[5:7].mean())*100))
print()
print(f"{'stopped on day':>15} | {'cumulative lift':>16} | {'that day alone':>15}")
print("-"*54)
for k in [1, 3, 7, 14, 28]:
    cum = (treat[:k].mean() - base[:k].mean())*100
    today = (treat[k-1] - base[k-1])*100
    print(f"{k:>15} | {cum:>+15.2f}pp | {today:>+14.2f}pp")

Two separate lessons here.

**Run whole weeks.** The baseline swings **2.1pp** between weekdays and weekends. A
Monday-to-Friday test measures a different population from a seven-day test, and comparing
across tests of different lengths is meaningless.

**Plot the daily lift, not just the cumulative one.** By day 14 the *daily* lift is +0.02pp
— the novelty is gone — but the **cumulative** figure still reads +0.45pp because it is
still averaging in week one. Almost every experiment dashboard shows only the cumulative
line, which is precisely the one that hides this.

> If the daily lift line is still falling at the end of your test, you have not yet measured
> the long-run effect of the change. You have measured people reacting to something new.

---
# 4 · Analysing it

## 4.1 · The test itself

For a conversion rate, a two-proportion z-test. Note the two different standard errors.

In [ ]:
def two_prop_full(c_a, n_a, c_b, n_b, alpha=0.05):
    p_a, p_b = c_a/n_a, c_b/n_b
    diff = p_b - p_a                                   # ABSOLUTE lift

    # pooled SE - used for the TEST, which assumes H0 is true
    pbar = (c_a + c_b) / (n_a + n_b)
    z = diff / np.sqrt(pbar*(1-pbar)*(1/n_a + 1/n_b))
    p = 2*(1 - stats.norm.cdf(abs(z)))

    # unpooled SE - used for the INTERVAL, which does not assume H0
    se = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
    c  = stats.norm.ppf(1 - alpha/2)
    return dict(p_a=p_a, p_b=p_b, abs_lift=diff, rel_lift=diff/p_a,
                z=z, p=p, ci=(diff - c*se, diff + c*se))

r = two_prop_full(1483, 14900, 1712, 14900)
for k, v in r.items():
    print(f"  {k:>9} : {v}")

**Two details worth the minute:**

- **Pooled vs unpooled SE.** The *test* assumes the null is true, so under that assumption
  both arms share one rate and you pool them. The *interval* makes no such assumption, so it
  must not pool. Using one for both is a common and very quiet error.
- **Absolute vs relative.** "+1.54 percentage points" and "+15.4%" are the same result.
  Pick one, label it explicitly, and never mix them in a sentence. "Conversion went up 15%"
  is ambiguous and has started real arguments.

For a continuous metric — time on page, items per order — use
`stats.ttest_ind(a, b, equal_var=False)`. Day 28's rule still holds: **Welch, always.**

## 4.2 · ⚠️ Revenue metrics

Revenue per user is heavy-tailed: most users spend little, a few spend enormously. What does
that do to a test?

In [ ]:
rng = np.random.default_rng(3008)
rev = rng.lognormal(1.0, 1.8, 100_000)
cut = np.quantile(rev, 0.99)
print(f"median {np.median(rev):.2f}   mean {rev.mean():.2f}   max {rev.max():,.0f}")
print("the top 1%% of users carry %.1f%% of all revenue" % (rev[rev >= cut].sum()/rev.sum()*100))

In [ ]:
# Does the heavy tail break the false-positive rate?  (both arms identical)
rng = np.random.default_rng(3004)
TR = 3000
print("FALSE POSITIVE RATE (should be 5%):")
for n in [200, 1000, 5000]:
    bad = sum(stats.ttest_ind(rng.lognormal(1.0, 1.8, n),
                              rng.lognormal(1.0, 1.8, n),
                              equal_var=False).pvalue < 0.05 for _ in range(TR))
    print(f"   n = {n:>5,}/arm   {bad/TR:>6.2%}")

**It does not.** The measured rate is 3–4%, *below* the nominal 5% — the Welch test is
mildly conservative here, not anti-conservative. The common claim that heavy tails "break"
the t-test's error rate is, in this direction, wrong.

The real damage is to **power**.

In [ ]:
# Now give the treatment a REAL +10% lift and see whether we can detect it
rng = np.random.default_rng(3006)
TR = 3000
print("POWER to detect a real +10% revenue lift:")
print(f"{'n per arm':>10} | {'raw':>7} | {'top 1% capped':>14}")
print("-"*36)
for n in [1000, 5000, 20000]:
    raw = cap_ = 0
    for _ in range(TR):
        a = rng.lognormal(1.0, 1.8, n)
        b = rng.lognormal(1.0, 1.8, n) * 1.10
        if stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05: raw += 1
        cap = np.quantile(np.concatenate([a, b]), 0.99)
        if stats.ttest_ind(np.minimum(a, cap), np.minimum(b, cap),
                           equal_var=False).pvalue < 0.05: cap_ += 1
    print(f"{n:>10,} | {raw/TR:>6.1%} | {cap_/TR:>13.1%}")

In [ ]:
# For comparison: the same +10% lift on a well-behaved metric
rng = np.random.default_rng(3007)
for n in [1000, 5000]:
    good = sum(stats.ttest_ind(rng.normal(10, 6, n), rng.normal(11, 6, n),
                               equal_var=False).pvalue < 0.05 for _ in range(3000))
    print(f"well-behaved metric, n = {n:,}/arm  ->  power {good/3000:.1%}")

A real **+10%** lift, at 1,000 users per arm:

- well-behaved metric → **96%** power
- revenue → **9%** power

You would miss it nine times out of ten. The test is not lying to you; **it simply cannot
see anything**, because a handful of users dominate the variance.

**What helps:**

- **Cap the top 1%** (decided in advance, written in the plan). Power at 20,000/arm goes from
  56% to **94%**. You are trading a little bias for a lot of precision, deliberately.
- **Test conversion rate instead**, and treat revenue as a secondary metric.
- **Plan for a much larger n** if revenue genuinely has to be the primary metric.

⚠️ Capping *after* you have seen the result, or trying several caps and keeping the best, is
p-hacking. The cap goes in the plan.

## 4.3 · ⚠️ Segments

The test came back flat. Somebody says *"but did it work for mobile users?"*

In [ ]:
rng = np.random.default_rng(3003)
TRIALS, hits = 4000, 0
for _ in range(TRIALS):
    found = False
    for _s in range(8):                      # eight segments, each 2,000 per arm
        n = 2000
        ca = rng.binomial(n, 0.10); cb = rng.binomial(n, 0.10)   # NO effect anywhere
        if two_prop(ca, n, cb, n)[1] < 0.05:
            found = True
    hits += found

print("8 segments, no real effect anywhere")
print("  simulated chance of at least one 'winning' segment: %.1f%%" % (hits/TRIALS*100))
print("  theory, 1 - 0.95^8                                : %.1f%%" % ((1-0.95**8)*100))
print()
for k in [1, 4, 8, 12]:
    print(f"  {k:>2} segments -> {(1-0.95**k)*100:>4.0f}% chance of a false winner")

Day 28 and Day 29 in a new costume. Eight segments gives a **34%** chance of finding a
"winner" in data with nothing in it; twelve gives 46%.

**The rule:** segments named in the plan *before* the test are hypotheses and can be tested.
Segments found *afterwards* are observations — report them as "worth a follow-up test",
never as a result, and say how many you looked at.

This is the single most common way a dead A/B test gets resurrected in a meeting.

---
# 5 · Deciding and reporting

## 5.1 · ⭐ "Not significant" is three different results

Suppose your team's threshold is **+0.5pp** — below that, shipping is not worth the
engineering cost. Here are three tests, none significant.

In [ ]:
rng = np.random.default_rng(3030)
print(f"{'n per arm':>10} | {'lift':>8} | {'p':>6} | {'95% CI':>22} | what you learned")
print("-"*95)
for n in [1200, 6000, 60000]:
    ca = rng.binomial(n, 0.10); cb = rng.binomial(n, 0.10)
    diff, p, (lo, hi) = two_prop(ca, n, cb, n)
    if hi > 0.005 and lo < -0.005:
        msg = "nothing - a big win and a big loss are both still possible"
    elif hi < 0.005 and lo > -0.005:
        msg = "a real answer: anything worth shipping is ruled out"
    else:
        msg = "still nothing - a worthwhile win is compatible with this"
    print(f"{n:>10,} | {diff*100:>+7.3f}pp | {p:>6.2f} | "
          f"[{lo*100:>+6.2f}, {hi*100:>+6.2f}]pp | {msg}")

Same verdict, three completely different amounts of information.

- **n = 1,200** — the interval allows a +2pp win *and* a −2.8pp disaster. You spent two weeks
  and learned nothing at all.
- **n = 6,000** — p = 0.08, and a +2pp win is still entirely compatible with the data. This
  is the one people wrongly report as "no effect".
- **n = 60,000** — anything above +0.43pp is ruled out. That is below the ship threshold, so
  the answer is *do not ship* — and that is a genuine finding, not a failure.

> **"Not significant" means "we did not detect an effect", never "there is no effect".**
> Only the interval can tell you which of the three you are looking at.

## 5.2 · The decision table

| Where the 95% interval sits | What it means | Decision |
|---|---|---|
| Entirely above the ship threshold | a win big enough to be worth it | **SHIP** |
| Crosses the threshold, excludes zero | real, but maybe too small to pay for | judgement call — weigh the cost |
| Entirely inside ± the threshold | any effect is too small to matter | **KILL**, and you learned something |
| Wide, crossing everything | the test was too small | **INCONCLUSIVE** — not "no effect" |
| Entirely below zero | the change actively hurt | **KILL**, and understand why |

**Statistical significance ≠ practical significance.** A significant +0.05pp on ten million
users is real and not worth the engineering. Decide the threshold *before* the test, with the
people who pay for it.

**And a kill is a success.** Most tested ideas do not work. A well-powered test that rules
out anything worth shipping has saved the team months — that is the common outcome of
experimentation, not the failure case.

## 5.3 · A full worked example

A checkout redesign, start to finish.

In [ ]:
# --- PLANNED, before any data existed ---
baseline, mde = 0.10, 0.01
need = n_per_arm(baseline, mde)
print(f"plan: baseline {baseline:.0%}, smallest worthwhile lift +{mde*100:.0f}pp")
print(f"      -> {need:,} per arm  ->  {need/2000:.1f} days at 2,000/day/arm")
print("      -> round up to 14 days, two whole weeks\n")

# --- WHAT CAME BACK ---
rng = np.random.default_rng(3010)
n_a = n_b = 14_900
c_a = rng.binomial(n_a, 0.1000)      # control, truly 10.00%
c_b = rng.binomial(n_b, 0.1120)      # treatment, truly 11.20%
print("observed: control %d/%d, treatment %d/%d" % (c_a, n_a, c_b, n_b))

In [ ]:
# --- CHECK THE SPLIT FIRST ---
tot = n_a + n_b
srm = stats.chisquare([n_a, n_b], [tot/2, tot/2])
print("SRM check: chi2 = %.2f, p = %.3f -> %s\n"
      % (srm.statistic, srm.pvalue, "clean" if srm.pvalue > 0.01 else "BUG"))

# --- THEN ANALYSE ---
r = two_prop_full(c_a, n_a, c_b, n_b)
print("control    %.3f%%" % (r["p_a"]*100))
print("treatment  %.3f%%" % (r["p_b"]*100))
print("abs lift   %+.3fpp   95%% CI [%+.3f, %+.3f]pp"
      % (r["abs_lift"]*100, r["ci"][0]*100, r["ci"][1]*100))
print("rel lift   %+.2f%%" % (r["rel_lift"]*100))
print("z = %.3f, p = %.3e" % (r["z"], r["p"]))
print("\n(the true lift built into this simulation was +1.20pp)")

Worth pausing on that last line. The test measured **+1.54pp** when the truth was
**+1.20pp**. It is well powered and correctly run, and it still landed a third of a
percentage point off — comfortably inside its own interval of +0.83 to +2.24pp.

**That is what the interval is for.** A point estimate is never the answer; the range is.

In [ ]:
# --- SECONDARY METRIC: revenue per user ---
rng = np.random.default_rng(3011)
rev_a = rng.lognormal(1.0, 1.8, n_a)
rev_b = rng.lognormal(1.0, 1.8, n_b) * 1.05
t = stats.ttest_ind(rev_a, rev_b, equal_var=False)
print("revenue/user: %.3f -> %.3f  (%+.3f), Welch p = %.4f"
      % (rev_a.mean(), rev_b.mean(), rev_b.mean()-rev_a.mean(), t.pvalue))

cap = np.quantile(np.concatenate([rev_a, rev_b]), 0.99)
t2 = stats.ttest_ind(np.minimum(rev_a, cap), np.minimum(rev_b, cap), equal_var=False)
print("with the top 1%% capped:                       p = %.4f" % t2.pvalue)
print("\nSECONDARY metric - supportive, but it cannot carry the decision.")

## 5.4 · The report

> *"We tested a checkout redesign against the current page, randomised by user, 14,900 users
> per arm over 14 days (planned for 14,751 to detect +1pp at 80% power). Checkout conversion
> rose from 9.95% to 11.49%: an absolute lift of +1.54pp, 95% CI +0.83 to +2.24pp
> (p = 1.8e-05), a relative lift of +15.4%. The split passed an SRM check. Revenue per user
> rose +2.33 (p = 0.033), which we treat as directional. The interval sits entirely above our
> +0.5pp ship threshold, so we recommend shipping."*

Eighty words, and it contains: what changed, the unit, n per arm and the planned n, both
rates, the absolute lift with its interval, the relative lift labelled as such, the exact p,
the SRM check, the secondary metric marked as secondary, and the decision with the threshold
it was judged against.

And one more line that is harder to write: **"we looked at N segments / ran M tests this
quarter."** Nothing else in the report tells the reader how much searching went into it.

---
# 6 · Putting it together

One function that does the whole analysis and refuses to proceed when the split is broken.

In [ ]:
def ab_test(c_a, n_a, c_b, n_b, ship_threshold=0.005, alpha=.05):
    """Analyse a conversion A/B test and recommend a decision."""

    #  0. SRM first - if the split is broken, nothing else matters
    tot = n_a + n_b
    srm_p = stats.chisquare([n_a, n_b], [tot/2, tot/2]).pvalue
    if srm_p < 0.01:
        return print(f"STOP: sample ratio mismatch (p={srm_p:.1e}). "
                     f"Fix the bug and rerun - do not analyse this.")

    p_a, p_b = c_a/n_a, c_b/n_b
    diff = p_b - p_a
    pbar = (c_a + c_b) / tot                          # pooled, for the test
    z = diff / np.sqrt(pbar*(1-pbar)*(1/n_a + 1/n_b))
    p = 2*(1 - stats.norm.cdf(abs(z)))
    se = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)   # unpooled, for the CI
    lo, hi = diff - 1.96*se, diff + 1.96*se

    print(f"control {p_a:.3%} ({c_a:,}/{n_a:,}) | treat {p_b:.3%} ({c_b:,}/{n_b:,})")
    print(f"lift {diff*100:+.3f}pp  95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp")
    print(f"relative {diff/p_a:+.2%}   p = {p:.3e}   SRM p = {srm_p:.2f}")

    t = ship_threshold
    verdict = ("SHIP"          if lo >  t else
               "KILL - hurts"  if hi <  0 else
               "KILL - too small to matter" if (lo > -t and hi < t) else
               "JUDGEMENT CALL - real but maybe too small" if lo > 0 else
               "INCONCLUSIVE - underpowered, not 'no effect'")
    print("->", verdict)

In [ ]:
print("=== the worked example ===")
ab_test(1483, 14900, 1712, 14900)

print("\n=== a broken split ===")
ab_test(2000, 20600, 2050, 19400)

print("\n=== an underpowered flat test ===")
ab_test(120, 1200, 115, 1200)

print("\n=== big, tight, and below the threshold ===")
ab_test(6000, 60000, 6053, 60000)

Note that the function **refuses** to analyse an SRM-failing test rather than printing a
warning next to the numbers. A warning gets skipped; a refusal does not.

What `ab_test()` cannot do is size the test before it runs. That is `n_per_arm()`, and it is
the half that matters most.

---
# Recap — today in twelve lines

1. ⭐ A coin flip removes **confounding, reverse causation and selection** at once.
2. It does **not** remove chance — that is what the p-value and the interval are for.
3. Decide three things first: the **unit**, **one** primary metric, and the **size**.
4. Randomise by **user**. By pageview, one person sees both arms and the p-value is wrong.
5. Halve the effect you want to catch → roughly **4×** the users. +1pp = 14,751 per arm.
6. ⚠️ At 11% power, winning tests reported a +1.00pp lift as **+3.32pp**.
7. ⚠️ Peeking 20 times turns a 5% false-positive rate into **25%**.
8. ⚠️ A 51.5/48.5 split on 40,000 users is p = 2e-09. That is a bug, not luck — rerun it.
9. ⚠️ Novelty: day 1 read +1.79pp; by day 14 the daily lift was **+0.02pp**.
10. ⚠️ A real +10% revenue lift had **9%** power at n = 1,000. Cap the top 1% in advance.
11. ⚠️ Eight segments on a flat test → **34%** chance of a fake winner.
12. Decide from the **interval** against a **threshold**. "Not significant" ≠ "no effect".

## Homework

- Write a one-page test plan for a change you would actually make: the unit, the primary
  metric, the MDE, n per arm, and the end date.
- Find a published A/B result and work out what is missing from the report.
- Take a feature you believe works and name the smallest lift that would justify building it.

---

## Module 2 ends here

Day 23 gave you centre, spread and shape. Day 24 made the standard deviation into a ruler.
Days 25 and 26 counted events. Day 27 explained why the bell curve keeps appearing. Day 28
turned a standard error into a decision. Day 29 showed what a correlation can and cannot
tell you.

Today used all of it on a single question somebody has to answer on Monday.

### Next — Topic 2.20: the statistics mini project
Hypothesis testing on business data, then the Module 2 assessment. After that, Module 3 —
Machine Learning, where today's habit of *deciding the rule before you look at the data*
becomes train/test discipline.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*